<img src="https://www.th-koeln.de/img/logo.svg" style="float:right;" width="200">

# 4th exercise: <font color="#C70039">Use Isolation Forest for anomaly detection</font>
* Course: AML
* Lecturer: <a href="https://www.gernotheisenberg.de/">Gernot Heisenberg</a>
* Author of notebook: <a href="https://www.gernotheisenberg.de/">Gernot Heisenberg</a>
* Date:   01.08.2026

<img src="https://upload.wikimedia.org/wikipedia/commons/c/ce/Isolating_a_Non-Anomalous_Point.png" style="float: center;" width="450">

---------------------------------
**GENERAL NOTE 1**: 
Please make sure you are reading the entire notebook, since it contains a lot of information on your tasks (e.g. regarding the set of certain parameters or a specific computational trick), and the written mark downs as well as comments contain a lot of information on how things work together as a whole. 

**GENERAL NOTE 2**: 
* Please, when commenting source code, just use English language only. 
* When describing an observation please use English language, too
* This applies to all exercises throughout this course.  

---------------------

### <font color="ce33ff">DESCRIPTION</font>:
This notebook allows you for using the Isolation Forest algorithm for anomaly detection. Isolation Forest is an unsupervised learning algorithm that belongs to the ensemble decision trees family. The following <a href="https://cs.nju.edu.cn/zhouzh/zhouzh.files/publication/icdm08b.pdf">paper</a> explains the details on its theory and implementation. 

-------------------------------------------------------------------------------------------------------------

### <font color="FFC300">TASKS</font>:
The tasks that you need to work on within this notebook are always indicated below as bullet points. 
If a task is more challenging and consists of several steps, this is indicated as well. 
Make sure you have worked down the task list and commented your doings. 
This should be done by using markdown.<br> 
<font color=red>Make sure you don't forget to specify your name and your matriculation number in the notebook.</font>

**YOUR TASKS in this exercise are as follows**:
1. import the notebook to Google Colab or use your local machine.
2. make sure you specified your name and your matriculation number in the header below my name and date. 
    * set the date too and remove mine.
3. read the entire notebook carefully 
    * add comments wherever you feel it necessary for better understanding
    * run the notebook for the first time. 

4. take the three data sets from exercise 1 and apply the isolation forest to them.
5. implement an appropriate visualisation (chart) that renders the result (anomaly={yes,no}) for every data point TOGETHER with the original data point in your data set.
-----------------------------------------------------------------------------------

### First example

In [ ]:
# Google Colab setup: make repository files available under the expected relative paths.
import os
import subprocess
import sys

if "google.colab" in sys.modules:
    repository = "/content/AML"
    if not os.path.isdir(repository):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/gheisenberg/AML.git", repository], check=True)
    os.chdir(repository)

In [ ]:
# Import the Isolation Forest estimator and create a reproducible two-feature data set.
# Each row of random_data is one observation that the model can isolate.
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest

rng = np.random.default_rng(1)
random_data = rng.normal(loc=20.0, scale=20.0, size=(50_000, 2))

This code will output the predictions for each data point in an array. If the result is -1, it means that this specific data point is an outlier. If the result is 1, then it means that the data point is not an outlier.

In [ ]:
# fit_predict trains the forest and immediately returns one label per observation.
# A label of -1 means outlier; a label of 1 means inlier.
# Fit the estimator and return one label per training observation.
isolation_forest_1 = IsolationForest(
    max_samples=100,
    contamination="auto",
    random_state=1,
    n_jobs=-1,
)
outlier_labels = isolation_forest_1.fit_predict(random_data)

print(outlier_labels[:200])
print(f"Detected outliers: {np.count_nonzero(outlier_labels == -1)}")

### Second example

In [ ]:
# Build two dense groups so low-density regions are easy to interpret visually.
# The following histogram is the reference distribution for the anomaly model.
# Create and visualize a bimodal data distribution.
bimodal_data = np.concatenate(
    (
        rng.normal(loc=-2.0, scale=0.5, size=500),
        rng.normal(loc=2.0, scale=0.5, size=500),
    )
)

plt.hist(bimodal_data, bins=30, density=True)
plt.xlim(-5, 5)
plt.title("Bimodal data distribution")
plt.show()

Note, that there are three regions where the data has low probability to appear: 
* one on the right side of the distribution
* another one on the left
* and another around zero. 

Let's see if the IsolationForest is able to identify these three regions

In [ ]:
# Evaluate anomaly scores on a regular grid instead of only on training points.
# This makes the predicted outlier regions visible on the plot.
# Fit a reproducible Isolation Forest with 100 estimators.
isolation_forest_2 = IsolationForest(n_estimators=100, random_state=1)
isolation_forest_2.fit(bimodal_data.reshape(-1, 1))

# Evaluate the anomaly score over a regular grid on the x-axis.
score_grid = np.linspace(-6, 6, 200).reshape(-1, 1)
anomaly_score = isolation_forest_2.decision_function(score_grid)
outlier_labels = isolation_forest_2.predict(score_grid)

plt.plot(score_grid[:, 0], anomaly_score, label="anomaly score")
plt.fill_between(
    score_grid[:, 0],
    anomaly_score.min(),
    anomaly_score.max(),
    where=outlier_labels == -1,
    color="#FF00FF",
    alpha=0.4,
    label="outlier region",
)
plt.legend()
plt.ylabel("anomaly score")
plt.xlabel("x")
plt.xlim(-5, 5)
plt.show()